## Prepare the data

In [2]:
import json
import requests

# 🔹 Replace with YOUR GitHub file URL
github_url = "https://github.com/UruslKhan21/personalize_chatbot/blob/main/output/fine_tuning/data/fine_tuning.json"

# 🔹 Convert GitHub link → raw link
raw_url = github_url.replace("github.com/", "raw.githubusercontent.com/").replace("/blob/", "/")

# 🔹 Fetch file
response = requests.get(raw_url)
response.raise_for_status()  # throws error if file not found

# 🔹 Load JSON
data = json.loads(response.text)

print("Data loaded successfully!")
print(data)


Data loaded successfully!
['<|startoftext|>Person 1<|separator|>Hey\nHow are you?\n[26/02/2025, 9:18 AM] ~ Person 2: Hey! I’m good, what about you?\n[26/02/2025, 9:20 AM] ~ Person 1: I’m good too, just a bit tired<|endoftext|>', '<|startoftext|>Person 2<|separator|>Long night?\n[26/02/2025, 9:22 AM] ~ Person 1: Yeah, had to finish some work<|endoftext|>', '<|startoftext|>Person 1<|separator|>It took longer than expected<|endoftext|>', '<|startoftext|>Person 2<|separator|>I get that, I slept late too\n[26/02/2025, 9:25 AM] ~ Person 1: What were you doing?\n[26/02/2025, 9:26 AM] ~ Person 2: Watching a show, then got stuck on YouTube 😂<|endoftext|>', '<|startoftext|>Person 1<|separator|>Hahaha classic, which show?<|endoftext|>', '<|startoftext|>Person 2<|separator|>Breaking Bad, ever watched it?\n[26/02/2025, 9:30 AM] ~ Person 1: Oh yeah, one of my favorites!<|endoftext|>', '<|startoftext|>Person 1<|separator|>It just keeps getting better\n[26/02/2025, 9:32 AM] ~ Person 2: Same! I’m at se

### 2. Load the tokenizer

In [3]:
import sys
sys.path.insert(0, "/content/personalize_chatbot")

In [4]:
import requests
import os

# ✅ 1. Use the RAW GitHub URL
raw_url = "https://raw.githubusercontent.com/UruslKhan21/personalize_chatbot/main/output/tokenizer/my_tokenizer.model"

# ✅ 2. Download tokenizer into the current directory
response = requests.get(raw_url)
response.raise_for_status()

with open("my_tokenizer.model", "w", encoding="utf-8") as f:
    f.write(response.text)

print("Downloaded tokenizer ✔")

# ✅ 3. Correct import (your tokenizer is in minbpe/regex.py)
from minbpe.regex import RegexTokenizer

# Create tokenizer instance
tokenizer = RegexTokenizer()

# ✅ 4. Load tokenizer from the file you downloaded
tokenizer.load("my_tokenizer.model")

# Function to get vocab size
def get_vocab_size(tokenizer: RegexTokenizer) -> int:
    return len(tokenizer.vocab)

print("Vocab size:", get_vocab_size(tokenizer))




Downloaded tokenizer ✔


ModuleNotFoundError: No module named 'minbpe'

### 3. Tokenize the sequences

In [ ]:
tokenized_data = []
for item in data:
    tokenized_item = tokenizer.encode(item, allowed_special="all")
    tokenized_data.append(tokenized_item)

len(tokenized_data[0])

131

### 4. Split the data

We need to be careful when splitting the data. We want to keep the multi-turn conversations complete in each part. So, the training and validation sets should start with a `You` message and end with an `Assistant` message.

In [ ]:
initial_split_index = int(0.95 * len(data))

# Adjusting the index to ensure that the training set ends with "Assistant" message
# and that the validation set starts with "You" message

# Scanning backward to find an Assistant message
# ---- CONFIG ----
SEPARATOR = "<|separator|>"
START_ASSISTANT = "<|startoftext|>Assistant"
START_USER = "<|startoftext|>You"

# ---- HELPERS ----
def safe_preview(line: str, sep: str = SEPARATOR, maxlen: int = 120) -> str:
    if not isinstance(line, str):
        return "[NON-STRING LINE]"
    if sep in line:
        return line.split(sep, 1)[0][:maxlen]
    return f"[NO '{sep}' FOUND] {line[:maxlen]}"

def print_previews(name: str, lines: list):
    print(f"\n{name} set:")
    if not lines:
        print("  (empty)")
        return
    print(f"  Count: {len(lines)}")
    print(f"  Start message: {safe_preview(lines[0])}")
    print(f"  End message:   {safe_preview(lines[-1])}")

# ---- CLEAN INPUT ----
# Ensure 'data' is a list of non-empty strings
data = [str(d).strip() for d in data if str(d).strip()]
n = len(data)

if n == 0:
    raise ValueError("Your dataset 'data' is empty after cleaning.")

# ---- FIND SPLIT ----
initial_split_index = max(1, min(int(0.95 * n), n - 1))

# Try to end train on an Assistant message by scanning backward
split_index = initial_split_index
while split_index > 0 and not data[split_index - 1].startswith(START_ASSISTANT):
    split_index -= 1

# If we never found an Assistant boundary, fall back to the 95% split
if split_index == 0:
    split_index = initial_split_index

# Final safety: ensure both sides non-empty
if split_index <= 0:
    split_index = 1
if split_index >= n:
    split_index = n - 1

train_data = data[:split_index]
val_data = data[split_index:]

# ---- OPTIONAL: ensure val starts with a User message; if not, nudge forward
if val_data and not val_data[0].startswith(START_USER):
    # scan forward within a small window to find a You line
    found = False
    for i in range(min(50, len(val_data))):
        if val_data[i].startswith(START_USER):
            train_data = data[:split_index + i]
            val_data = data[split_index + i:]
            found = True
            break
    # if not found, we proceed as-is (no crash)

# ---- PRINT DEBUG + PREVIEWS ----
print(f"Total lines: {n}")
print(f"Split index: {split_index} (train={len(train_data)}, val={len(val_data)})")

print_previews("Training", train_data)
print_previews("Validation", val_data)


Total lines: 51
Split index: 48 (train=48, val=3)

Training set:
  Count: 48
  Start message: <|startoftext|>Person 1
  End message:   <|startoftext|>Person 2

Validation set:
  Count: 3
  Start message: <|startoftext|>Person 1
  End message:   <|startoftext|>Person 1


We got the index that we should use to split the data. Now, let's split the tokenized data.

In [5]:
train_data = tokenized_data[:split_index]
val_data = tokenized_data[split_index:]

NameError: name 'tokenized_data' is not defined

Now, we need to combine the `You` and `Assistant` turns into one sequence. We will make sure that the resulting sequence does not exceed the `block_size`.

In [ ]:
block_size = 256


def combine_turns(data: list[list[int]], should_trim_long_sequences: bool) -> list[list[int]]:
    combined_turns_data = []
    for i in range(0, len(data)-1, 2):
        you_message = data[i]
        assistant_message = data[i+1]
        if not you_message or not assistant_message:
            continue

        final_message = you_message + assistant_message
        if len(final_message) > block_size and should_trim_long_sequences:
            final_message = final_message[-block_size:]

        combined_turns_data.append(final_message)
    return combined_turns_data


combined_train_data = combine_turns(
    data=train_data,
    should_trim_long_sequences=True
)
combined_val_data = combine_turns(
    data=val_data,
    should_trim_long_sequences=True
)

In [ ]:
print("Train data")
print(f"Length before: {len(train_data)}")
print(f"Length after: {len(combined_train_data)}")

print("\nValidation data")
print(f"Length before: {len(val_data)}")
print(f"Length after: {len(combined_val_data)}")

Train data
Length before: 48
Length after: 24

Validation data
Length before: 3
Length after: 1


Let's convert each sequence of tokens into a tensor.

In [6]:
import torch

train_data = torch.tensor(combined_train_data)
val_data = torch.tensor(combined_val_data)

NameError: name 'combined_train_data' is not defined

Since our token sequences don't all have the same length, we can't turn the data into a tensor all at once. To do that, all sequences need to have the same length.

That's why we need to use padding to fix this problem. We can add padding at the start or end of the sequence. Let's add it to the start.

In [ ]:
import torch
torch.manual_seed(3647)

# The token `<|padding|>` is used to mask the padding tokens.
# Masking means the model will ignore these tokens during training.
# In other words, the loss will not be calculated for these tokens.
padding_token = tokenizer.special_tokens["<|padding|>"]


def apply_padding_to_data(data: list[list[int]], block_size: int, padding_token: int) -> torch.Tensor:
    tensors = []
    for i in range(len(data)):
        tensor = torch.tensor(data[i])
        padded_tensor = torch.nn.functional.pad(
            input=tensor,
            # for right padding:
            pad=(0, block_size - len(tensor)),
            # pad=(block_size - len(tensor), 0),
            value=padding_token
        )
        tensors.append(padded_tensor)

    return torch.stack(tensors)


train_data_tensor = apply_padding_to_data(
    data=combined_train_data,
    block_size=block_size,
    padding_token=padding_token
)
val_data_tensor = apply_padding_to_data(
    data=combined_val_data,
    block_size=block_size,
    padding_token=padding_token
)

train_data_tensor.shape, val_data_tensor.shape

(torch.Size([24, 256]), torch.Size([1, 256]))

In [ ]:
train_data_tensor[0]

tensor([1024,   80,  292,  115,  281,   32,   49, 1025,   72,  101,  121,   10,
          72,  111,  119,   32,  294,  101,   32,  997,  659,   91,   50,   54,
          47,   48,   50,   47,   50,   48,   50,   53,   44,   32,   57,   58,
          49,   56,  729,   77,   93,   32,  126,   32,   80,  292,  115,  281,
          32,   50,   58,   32,   72,  101,  121,   33,   32,   73,  905,  109,
          32,  103, 1018,  100,   44,   32,  728,  273,   32,  344,  111,  487,
          32,  997,  659,   91,   50,   54,   47,   48,   50,   47,   50,   48,
          50,   53,   44,   32,   57,   58,   50,   48,  729,   77,   93,   32,
         126,   32,   80,  292,  115,  281,   32,   49,   58,   32,   73,  905,
         109,   32,  103, 1018,  100,   32,  432,  111,   44,   32,  106,  374,
         116,   32,   97,   32,   98,  325,   32,  462,  310,  100, 1026, 1024,
          80,  292,  115,  281,   32,   50, 1025,   76,  281,  103,   32,  667,
         685,  116,  659,   91,   50,   

In [ ]:
val_data_tensor[0]

tensor([1024,   80,  292,  115,  281,   32,   49, 1025,   89,  111,  117,   32,
         575,  310,   63,   32,   73,   32,   99,  269,   32,  100,  492,  970,
          32,  105,  102,   32,  997,  905,  310,   32,  462,  310,  100, 1026,
        1024,   80,  292,  115,  281,   32,   50, 1025,   78,   97,  104,   44,
          32,   73,   32,  103,  111,  116,   32,  295,  335, 1026, 1028, 1028,
        1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028,
        1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028,
        1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028,
        1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028,
        1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028,
        1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028,
        1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028, 1028,
        1028, 1028, 1028, 1028, 1028, 10

### 5. Creat the data loaders

In [ ]:
train_data_tensor.shape

torch.Size([24, 256])

In [ ]:
from typing import Tuple
from torch.utils.data import Dataset, DataLoader


class FineTuningDataset(Dataset):
    def __init__(self, data: torch.Tensor, device: torch.device, padding_token: int):
        self.data = data  # shape: (num_samples, block_size)
        self.device = device
        self.padding_token = padding_token

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        sample = self.data[index]
        x = sample.to(self.device)
        y = sample[1:].to(self.device)
        padding_tensor = torch.tensor([self.padding_token], device=self.device)
        y = torch.cat((y, padding_tensor))
        return x, y


batch_size = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = FineTuningDataset(
    data=train_data_tensor,
    device=device,
    padding_token=padding_token
)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_dataset = FineTuningDataset(
    data=val_data_tensor,
    device=device,
    padding_token=padding_token
)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
x, y = next(iter(train_loader))
x.shape, y.shape

(torch.Size([24, 256]), torch.Size([24, 256]))

## Fine-tuning

### 1. Load the saved checkpoint

In [ ]:
from transformer.model import GPTLanguageModel

block_size = 256
n_embd = 512
n_head = 8
n_layer = 4
dropout = 0.2
batch_size = 64
vocab_size = get_vocab_size(tokenizer)

model = GPTLanguageModel(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    n_layer=n_layer,
    dropout=dropout,
    device=device,
    ignore_index=tokenizer.special_tokens["<|padding|>"],
).to(device)
model = torch.compile(model)

print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

13.790213 M parameters


In [ ]:
import requests

url = "https://github.com/UruslKhan21/personalize_chatbot/blob/main/output/pre_training/run_4/checkpoint_0.pth"
resp = requests.get(url)
resp.raise_for_status()

with open("checkpoint_0.pth", "wb") as f:
    f.write(resp.content)
    


In [ ]:
import requests

# Raw GitHub URL (update if your folder name differs)
ckpt_url = "https://github.com/UruslKhan21/personalize_chatbot/blob/main/output/pre_training/run_4/checkpoint_0.pth"

# Download file into Colab
response = requests.get(ckpt_url)
response.raise_for_status()

with open("checkpoint_0.pth", "wb") as f:
    f.write(response.content)

print("✓ Checkpoint downloaded as checkpoint_0.pth")


✓ Checkpoint downloaded as checkpoint_0.pth


In [ ]:
import os, requests, torch

# 1) Use the RAW GitHub URL (not the /blob/ page)
CKPT_URL = "https://raw.githubusercontent.com/UruslKhan21/personalize_chatbot/main/output/pre_training/run_4/checkpoint_0.pth"

# 2) Download in BINARY and sanity-check
local_path = "checkpoint_0.pth"
r = requests.get(CKPT_URL, stream=True, timeout=120)
r.raise_for_status()

with open(local_path, "wb") as f:
    for chunk in r.iter_content(1024 * 1024):
        if chunk:
            f.write(chunk)

print("Saved:", local_path, "size:", os.path.getsize(local_path), "bytes")

# Quick sniff to make sure it isn't HTML or a Git LFS pointer
with open(local_path, "rb") as f:
    head = f.read(200)

if head.startswith(b"<") or b"<html" in head.lower():
    raise RuntimeError("Downloaded an HTML page (wrong URL). Use the RAW URL from raw.githubusercontent.com")

if head.startswith(b"version https://git-lfs.github.com/spec"):
    raise RuntimeError("This is a Git LFS pointer, not the real checkpoint. You must fetch via git-lfs, a release asset, or another host (Drive/HF).")

# 3) Load checkpoint (trusted)
ckpt = torch.load(local_path, map_location="cpu", weights_only=False)

# If you saved a dict with model_state_dict:
state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt

# ...construct your model first as `model = ...`
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Loaded. Missing:", missing, "Unexpected:", unexpected)


Saved: checkpoint_0.pth size: 5848894 bytes


RuntimeError: Error(s) in loading state_dict for OptimizedModule:
	size mismatch for _orig_mod.token_embedding_table.weight: copying a param with shape torch.Size([1034, 128]) from checkpoint, the shape in current model is torch.Size([1029, 512]).
	size mismatch for _orig_mod.position_embedding_table.weight: copying a param with shape torch.Size([64, 128]) from checkpoint, the shape in current model is torch.Size([256, 512]).
	size mismatch for _orig_mod.blocks.0.self_attention.heads.0.tril: copying a param with shape torch.Size([64, 64]) from checkpoint, the shape in current model is torch.Size([256, 256]).
	size mismatch for _orig_mod.blocks.0.self_attention.heads.0.key.weight: copying a param with shape torch.Size([16, 128]) from checkpoint, the shape in current model is torch.Size([64, 512]).
	size mismatch for _orig_mod.blocks.0.self_attention.heads.0.query.weight: copying a param with shape torch.Size([16, 128]) from checkpoint, the shape in current model is torch.Size([64, 512])

In [ ]:
checkpoint_path = "../output/pre_training/run_4/checkpoint_0.pth"
checkpoint = torch.load(checkpoint_path, weights_only=True)
model_state_dict = checkpoint["model_state_dict"]
model.load_state_dict(model_state_dict)

NameError: name 'rm' is not defined

Generate from the model to make sure that the weights were loaded correctly.

In [ ]:
input_tokens = tokenizer.encode("all good", allowed_special="all")
input_tokens = torch.tensor(
    input_tokens, dtype=torch.long).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model.generate(input_tokens=input_tokens, max_new_tokens=100)

print(tokenizer.decode(output[0].tolist()))

ffersafferिसbola Abhi  �🤣🤣message �chola sa�do Vo ght े सke vo cessa	ly th और �ेedOk neिdana moHappy BaZrha cho<|unk|>me o= �gya �ere tu it मेंe kya िफ़ाज़तTo Ba फ़�ने �onconyste

*is message was dele7�ी स3 ion kar  फw�MDऐ
वकीना अज़ाबुलtra]Kal logा फरमाश�ज़proबter 


### 2. Estimate loss

In [ ]:
from typing import Dict


@torch.no_grad()
def estimate_loss(
    model: torch.nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
) -> Dict[str, float]:
    output = {}
    model.eval()

    for split, loader in [('train', train_loader), ('val', val_loader)]:
        losses = []
        for x, y in loader:
            with torch.no_grad():
                _, loss = model(x, y)
            losses.append(loss.item())
        output[split] = sum(losses) / len(losses)

    model.train()
    return output

NameError: name 'torch' is not defined

### 3. Save checkpoints

In [ ]:
def save_checkpoint(
    model: GPTLanguageModel,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    loss: float,
    file_path: str = "checkpoint.pth"
) -> None:
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss
    }
    torch.save(checkpoint, file_path)

### 4. Training loop

In [ ]:
max_iters = 100
eval_interval = 20
learning_rate = 1e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
train_losses = []
val_losses = []

for iteration in range(max_iters):
    for batch_idx, (x_batch, y_batch) in enumerate(train_loader):
        # Evaluation
        if batch_idx % eval_interval == 0 or batch_idx == len(train_loader) - 1:
            losses = estimate_loss(
                model=model,
                train_loader=train_loader,
                val_loader=val_loader,
            )
            train_losses.append(losses['train'])
            val_losses.append(losses['val'])

            print(
                f"iteration {iteration} / step {batch_idx}: "
                f"train loss {losses['train']:.4f}, "
                f"val loss {losses['val']:.4f}"
            )

        # Training step
        logits, loss = model(x_batch, y_batch)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    # Save checkpoint
    save_checkpoint(
    model=model,
    optimizer=optimizer,
    epoch=iteration,
    loss=loss.item(),
    file_path=f"checkpoint_{iteration}.pth"
)


FileNotFoundError: [Errno 2] No usable temporary directory found in ['/tmp', '/var/tmp', '/usr/tmp', '/content']

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Evaluation Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Over Time")
plt.legend()
plt.grid()
plt.show()

NameError: name 'train_losses' is not defined

<Figure size 1000x500 with 0 Axes>

In [ ]:
def get_input_tokens(message: str) -> torch.Tensor:
    input_tokens = tokenizer.encode(
        f"<|startoftext|>{message}<|separator|>", allowed_special="all")
    input_tokens = torch.tensor(
        input_tokens, dtype=torch.long).unsqueeze(0).to(device)
    return input_tokens


user_message = "10 baja"
input_tokens = get_input_tokens(message=user_message)
model_answer = ""

model.eval()
while True:
    output_tokens = model.generate(input_tokens=input_tokens, max_new_tokens=1)
    last_generated_token = output_tokens[0, -1].item()
    if last_generated_token == tokenizer.special_tokens["<|endoftext|>"]:
        break

    input_tokens = torch.cat((input_tokens, output_tokens[:, -1:]), dim=1)
    model_answer += tokenizer.decode([last_generated_token])

    if len(output_tokens[0]) > block_size:
        input_tokens = input_tokens[:, -block_size:]

print(f"You: {user_message}")
print(f"Assistant: {model_answer}")

You: 10 baja
Assistant: � हमेंPers ec m ास�{thहुersthocompp0instrue w?’son ों cth की
son 
icff, व अता फरमाayPerson 2<|separator|>m w
ारी �h1upe as  cNahi  Pers
hai  �e bhayou Wckr k7ut� m$��
